# TPC-H: DuckDB vs Opteryx, each on its native format

Runs the 22 standard **TPC-H** queries against two embedded SQL engines. Each engine reads **its own native on-disk format** rather than a shared interchange format:

| Engine | Native format under test |
|---|---|
| **DuckDB** | its own single-file columnar database (`.duckdb`) |
| **Opteryx** | [`.skene`](https://github.com/mabel-dev/opteryx), the draken-native columnar file format used by Opteryx |

The pipeline is: generate TPC-H with DuckDB's built-in `dbgen`, export it to Parquet, convert the Parquet to `.skene` for Opteryx, run all 22 queries on both engines, and compare wall-clock time.

This notebook accompanies [Benchmarking](https://docs.opteryx.app/docs/reference/internals/benchmarking) on docs.opteryx.app. The numbers it produces belong to whatever machine runs it. A Colab runtime is small and shared, so compare the two engines with each other, not with figures published anywhere else.

**Runtime note:** at the default scale factor of 1, `lineitem` is about 6M rows and the whole run takes a few minutes on a Colab CPU runtime. For a quick smoke test, lower `SCALE_FACTOR` to `0.1`.

In [ ]:
# --- Configuration -----------------------------------------------------
SCALE_FACTOR = 1         # TPC-H scale factor (1 = ~8.66M rows total). DuckDB's
                         # dbgen also accepts fractions, e.g. 0.1, for a fast smoke test.
ITERATIONS = 3           # warm runs per query per engine; the minimum is reported
import os
WORK_DIR = os.path.abspath("/content/tpch_bench" if os.path.isdir("/content") else "./tpch_bench")

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print("Working directory:", os.getcwd())

cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print("CPUs available:", cpus)

## 0. Install engines

`duckdb` and `opteryx-core` install from PyPI. `opteryx-core` bundles the `skene` file-format module and `rugo`, the Parquet reader used in the conversion step.

In [ ]:
%pip install -q --upgrade duckdb opteryx-core pandas matplotlib

import duckdb
import rugo
import skene
import opteryx

print("duckdb  ", duckdb.__version__)
print("opteryx ", opteryx.__version__)
print("skene   ", skene.__version__)

## 1. Generate TPC-H. This is the DuckDB-native database

DuckDB's `tpch` extension generates the dataset directly into a file-backed connection. That file is what DuckDB is benchmarked against below, so it reads its own native columnar storage with no import step.

Each table is also exported to Parquet, which is the input for the `.skene` conversion in the next step.

In [ ]:
DUCKDB_PATH = os.path.join(WORK_DIR, "tpch_native.duckdb")
if os.path.exists(DUCKDB_PATH):
    os.remove(DUCKDB_PATH)

con = duckdb.connect(DUCKDB_PATH)
con.execute("INSTALL tpch; LOAD tpch;")
con.execute(f"CALL dbgen(sf={SCALE_FACTOR});")

PARQUET_DIR = os.path.join(WORK_DIR, "tpch")
os.makedirs(PARQUET_DIR, exist_ok=True)

TABLES = [r[0] for r in con.execute("SHOW TABLES").fetchall()]
print(f"{len(TABLES)} tables: {TABLES}")
for t in TABLES:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    output_path = os.path.join(PARQUET_DIR, f"{t}.parquet")
    con.execute(f"COPY {t} TO '{output_path}' (FORMAT PARQUET)")
    print(f"  {t:<10} {n:>12,} rows -> {output_path}")
con.close()

## 2. Convert Parquet to `.skene`, the Opteryx-native format

`.skene` is the draken-native columnar file format used by Opteryx. It round-trips draken vectors losslessly, including dictionary encoding, logical type refinements, zone maps and per-column statistics. Parquet either drops these or has to re-derive them on every read.

The conversion builds the same layout as the Opteryx TPC-H suite (`make tpch-sf1`, via `dev/parquet_to_skene.py`): **65,536-row row groups packed into one file per table**, lz4-compressed, with read acceleration on. `rugo.parquet.read_parquet` streams the Parquet as draken `Morsel`s, which are re-chunked to that row-group size and streamed into a `skene.SkeneWriter`. The data never passes through an intermediate Python object model.

The layout matters. Writing one small `.skene` file per Parquet row group also works, but every file has a fixed open-and-footer cost. Measured at SF1, that layout made the suite about a third slower.

In [ ]:
from draken.morsels.morsel import Morsel
from rugo.parquet import read_parquet
from opteryx.connectors import DiskConnector

ROWS_PER_ROW_GROUP = 65_536
SKENE_ROOT = os.path.join(WORK_DIR, "data", "tpch")
os.makedirs(SKENE_ROOT, exist_ok=True)


def combine(morsels):
    return morsels[0] if len(morsels) == 1 else Morsel.combine(morsels)


for t in TABLES:
    table_dir = os.path.join(SKENE_ROOT, t)
    os.makedirs(table_dir, exist_ok=True)
    out_path = os.path.join(table_dir, f"{t}-000.skene")
    writer = skene.SkeneWriter(
        read_acceleration=True,   # value ordering + statistics + zone maps
        codec="lz4",
        zstd_level=0,
        path=out_path,
        scratch_path=os.path.join(table_dir, ".stage"),
    )
    pending, pending_rows, rows_written, row_groups = [], 0, 0, 0
    with read_parquet(os.path.join(PARQUET_DIR, f"{t}.parquet")) as reader:
        for morsel in reader:
            pending.append(morsel)
            pending_rows += morsel.num_rows
            while pending_rows >= ROWS_PER_ROW_GROUP:
                merged = combine(pending)
                writer.add_row_group(merged.slice(0, ROWS_PER_ROW_GROUP))
                row_groups += 1
                rows_written += ROWS_PER_ROW_GROUP
                remainder = merged.num_rows - ROWS_PER_ROW_GROUP
                pending = [merged.slice(ROWS_PER_ROW_GROUP, remainder)] if remainder else []
                pending_rows = remainder
    if pending_rows:
        writer.add_row_group(combine(pending))   # the last row group is short
        row_groups += 1
        rows_written += pending_rows
    writer.finish()
    print(f"{t:<10} {rows_written:>12,} rows in {row_groups:>3} row groups -> {out_path}")

# Register the workspace root so `data.tpch.<table>` resolves under WORK_DIR/data/tpch/<table>.
opteryx.register_workspace("data", DiskConnector)

## 3. The 22 TPC-H queries

These are the standard de-parameterized TPC-H queries that Opteryx validates in its own performance suite (`tests/performance/tpch/opteryx/queries/`). They are written with a `testdata.tpch.<table>` placeholder prefix. The only per-engine change is that prefix. Everything else runs unchanged on both engines:

| | DuckDB | Opteryx |
|---|---|---|
| table prefix | stripped (bare table name) | rewritten to `data.tpch.` |

In [ ]:
QUERIES = {'Q01': "select\n    l_returnflag,\n    l_linestatus,\n    sum(l_quantity) as sum_qty,\n    sum(l_extendedprice) as sum_base_price,\n    sum(l_extendedprice * (1 - l_discount)) as sum_disc_price,\n    sum(l_extendedprice * (1 - l_discount) * (1 + l_tax)) as sum_charge,\n    avg(l_quantity) as avg_qty,\n    avg(l_extendedprice) as avg_price,\n    avg(l_discount) as avg_disc,\n    count(*) as count_order\nfrom\n    testdata.tpch.lineitem\nwhere\n    l_shipdate <= '1998-09-16'::DATE\ngroup by l_returnflag, l_linestatus\norder by l_returnflag, l_linestatus;", 'Q02': "with q2_min_ps_supplycost as\n(select\n    p_partkey as min_p_partkey,\n    min(ps_supplycost) as min_ps_supplycost\nfrom\n    testdata.tpch.part,\n    testdata.tpch.partsupp,\n    testdata.tpch.supplier,\n    testdata.tpch.nation,\n    testdata.tpch.region\nwhere\n    p_partkey = ps_partkey\n    and s_suppkey = ps_suppkey\n    and s_nationkey = n_nationkey\n    and n_regionkey = r_regionkey\n    and r_name = 'EUROPE'\ngroup by\n    p_partkey)\n\nselect\n    s_acctbal,\n    s_name,\n    n_name,\n    p_partkey,\n    p_mfgr,\n    s_address,\n    s_phone,\n    s_comment\nfrom\n    testdata.tpch.part,\n    testdata.tpch.supplier,\n    testdata.tpch.partsupp,\n    testdata.tpch.nation,\n    testdata.tpch.region,\n    q2_min_ps_supplycost\nwhere\n    p_partkey = ps_partkey\n    and s_suppkey = ps_suppkey\n    and p_size = 37\n    and p_type like '%COPPER'\n    and s_nationkey = n_nationkey\n    and n_regionkey = r_regionkey\n    and r_name = 'EUROPE'\n    and ps_supplycost = min_ps_supplycost\n    and p_partkey = min_p_partkey\norder by\n    s_acctbal desc,\n    n_name,\n    s_name,\n    p_partkey\nlimit 100;", 'Q03': "select\n    l_orderkey,\n    sum(l_extendedprice * (1 - l_discount)) as revenue,\n    o_orderdate,\n    o_shippriority\nfrom\n    testdata.tpch.customer,\n    testdata.tpch.orders,\n    testdata.tpch.lineitem\nwhere\n    c_mktsegment = 'BUILDING'\n    and c_custkey = o_custkey\n    and l_orderkey = o_orderkey\n    and o_orderdate < '1995-03-22'::DATE\n    and l_shipdate > '1995-03-22'::DATE\ngroup by\n    l_orderkey,\n    o_orderdate,\n    o_shippriority\norder by\n    revenue desc,\n    o_orderdate\nlimit 10;", 'Q04': "select\n    o_orderpriority,\n    count(*) as order_count\nfrom\n    testdata.tpch.orders as o\nwhere\n    o_orderdate >= '1996-05-01'::DATE\n    and o_orderdate < '1996-08-01'::DATE\n    and exists (\n        select\n            *\n        from\n            testdata.tpch.lineitem\n        where\n            l_orderkey = o.o_orderkey\n            and l_commitdate < l_receiptdate\n    )\ngroup by\n    o_orderpriority\norder by\n    o_orderpriority;", 'Q05': "select\n    n_name,\n    sum(l_extendedprice * (1 - l_discount)) as revenue\nfrom\n    testdata.tpch.customer,\n    testdata.tpch.orders,\n    testdata.tpch.lineitem,\n    testdata.tpch.supplier,\n    testdata.tpch.nation,\n    testdata.tpch.region\nwhere\n    c_custkey = o_custkey\n    and l_orderkey = o_orderkey\n    and l_suppkey = s_suppkey\n    and c_nationkey = s_nationkey\n    and s_nationkey = n_nationkey\n    and n_regionkey = r_regionkey\n    and r_name = 'AFRICA'\n    and o_orderdate >= '1993-01-01'::DATE\n    and o_orderdate < '1994-01-01'::DATE\ngroup by\n    n_name\norder by\n    revenue desc;", 'Q06': "select\n    sum(l_extendedprice * l_discount) as revenue\nfrom\n    testdata.tpch.lineitem\nwhere\n    l_shipdate >= '1993-01-01'::DATE\n    and l_shipdate < '1994-01-01'::DATE\n    and l_discount between 0.06 - 0.01 and 0.06 + 0.01\n    and l_quantity < 25;", 'Q07': "select\n    supp_nation,\n    cust_nation,\n    l_year,\n    sum(volume) as revenue\nfrom\n    (\n        select\n            n1.n_name as supp_nation,\n            n2.n_name as cust_nation,\n            EXTRACT(YEAR FROM l_shipdate) as l_year,\n            l_extendedprice * (1 - l_discount) as volume\n        from\n            testdata.tpch.supplier,\n            testdata.tpch.lineitem,\n            testdata.tpch.orders,\n            testdata.tpch.customer,\n            testdata.tpch.nation n1,\n            testdata.tpch.nation n2\n        where\n            s_suppkey = l_suppkey\n            and o_orderkey = l_orderkey\n            and c_custkey = o_custkey\n            and s_nationkey = n1.n_nationkey\n            and c_nationkey = n2.n_nationkey\n            and (\n                (n1.n_name = 'KENYA' and n2.n_name = 'PERU')\n                or (n1.n_name = 'PERU' and n2.n_name = 'KENYA')\n            )\n            and l_shipdate between '1995-01-01'::DATE and '1996-12-31'::DATE\n    ) as shipping\ngroup by\n    supp_nation,\n    cust_nation,\n    l_year\norder by\n    supp_nation,\n    cust_nation,\n    l_year;", 'Q08': "select\n    o_year,\n    sum(case\n        when nation = 'PERU' then volume\n        else 0\n    end) / sum(volume) as mkt_share\nfrom\n    (\n        select\n            EXTRACT(YEAR FROM o_orderdate) as o_year,\n            l_extendedprice * (1 - l_discount) as volume,\n            n2.n_name as nation\n        from\n            testdata.tpch.part,\n            testdata.tpch.supplier,\n            testdata.tpch.lineitem,\n            testdata.tpch.orders,\n            testdata.tpch.customer,\n            testdata.tpch.nation n1,\n            testdata.tpch.nation n2,\n            testdata.tpch.region\n        where\n            p_partkey = l_partkey\n            and s_suppkey = l_suppkey\n            and l_orderkey = o_orderkey\n            and o_custkey = c_custkey\n            and c_nationkey = n1.n_nationkey\n            and n1.n_regionkey = r_regionkey\n            and r_name = 'AMERICA'\n            and s_nationkey = n2.n_nationkey\n            and o_orderdate between '1995-01-01'::DATE and '1996-12-31'::DATE\n            and p_type = 'ECONOMY BURNISHED NICKEL'\n    ) as all_nations\ngroup by\n    o_year\norder by\n    o_year;", 'Q09': "select\n    nation,\n    o_year,\n    sum(amount) as sum_profit\nfrom\n    (\n        select\n            n_name as nation,\n            EXTRACT(YEAR FROM o_orderdate) as o_year,\n            l_extendedprice * (1 - l_discount) - ps_supplycost * l_quantity as amount\n        from\n            testdata.tpch.part,\n            testdata.tpch.supplier,\n            testdata.tpch.lineitem,\n            testdata.tpch.partsupp,\n            testdata.tpch.orders,\n            testdata.tpch.nation\n        where\n            s_suppkey = l_suppkey\n            and ps_suppkey = l_suppkey\n            and ps_partkey = l_partkey\n            and p_partkey = l_partkey\n            and o_orderkey = l_orderkey\n            and s_nationkey = n_nationkey\n            and p_name like '%plum%'\n    ) as profit\ngroup by\n    nation,\n    o_year\norder by\n    nation,\n    o_year desc;", 'Q10': "select\n    c_custkey,\n    c_name,\n    sum(l_extendedprice * (1 - l_discount)) as revenue,\n    c_acctbal,\n    n_name,\n    c_address,\n    c_phone,\n    c_comment\nfrom\n    testdata.tpch.customer,\n    testdata.tpch.orders,\n    testdata.tpch.lineitem,\n    testdata.tpch.nation\nwhere\n    c_custkey = o_custkey\n    and l_orderkey = o_orderkey\n    and o_orderdate >= '1993-07-01'::DATE\n    and o_orderdate < '1993-10-01'::DATE\n    and l_returnflag = 'R'\n    and c_nationkey = n_nationkey\ngroup by\n    c_custkey,\n    c_name,\n    c_acctbal,\n    c_phone,\n    n_name,\n    c_address,\n    c_comment\norder by\n    revenue desc\nlimit 20;", 'Q11': "with q11_part_tmp_cached as\n(select\n    ps_partkey,\n    sum(ps_supplycost * ps_availqty) as part_value\nfrom\n    testdata.tpch.partsupp,\n    testdata.tpch.supplier,\n    testdata.tpch.nation\nwhere\n    ps_suppkey = s_suppkey\n    and s_nationkey = n_nationkey\n    and n_name = 'GERMANY'\ngroup by ps_partkey)\n\n, q11_sum_tmp_cached as\n(select\n    sum(part_value) as total_value\nfrom\n    q11_part_tmp_cached)\n\nselect\n    ps_partkey, part_value as value\nfrom (\n    select\n        ps_partkey,\n        part_value,\n        total_value\n    from\n        q11_part_tmp_cached cross join q11_sum_tmp_cached\n) a\nwhere\n    part_value > total_value * 0.0001\norder by\n    value desc;", 'Q12': "select\n    l_shipmode,\n    sum(case\n        when o_orderpriority = '1-URGENT'\n            or o_orderpriority = '2-HIGH'\n            then 1\n        else 0\n    end) as high_line_count,\n    sum(case\n        when o_orderpriority <> '1-URGENT'\n            and o_orderpriority <> '2-HIGH'\n            then 1\n        else 0\n    end) as low_line_count\nfrom\n    testdata.tpch.orders,\n    testdata.tpch.lineitem\nwhere\n    o_orderkey = l_orderkey\n    and l_shipmode in ('REG AIR', 'MAIL')\n    and l_commitdate < l_receiptdate\n    and l_shipdate < l_commitdate\n    and l_receiptdate >= '1995-01-01'::DATE\n    and l_receiptdate < '1996-01-01'::DATE\ngroup by\n    l_shipmode\norder by\n    l_shipmode;", 'Q13': "SELECT\n  c_count,\n  Count(*) AS custdist\nFROM\n  (\n    SELECT\n      c_custkey,\n      Count(o_orderkey) AS c_count\n    FROM\n      testdata.tpch.customer\n      LEFT OUTER JOIN (\n        SELECT\n          *\n        FROM\n          testdata.tpch.orders\n        WHERE\n          o_comment NOT LIKE '%unusual%accounts%'\n      ) AS t ON c_custkey = t.o_custkey\n    GROUP BY\n      c_custkey\n  ) c_orders\nGROUP BY\n  c_count\nORDER BY\n  custdist DESC,\n  c_count DESC;", 'Q14': "select\n    100.00 * sum(case\n        when p_type like 'PROMO%'\n            then l_extendedprice * (1 - l_discount)\n        else 0\n    end) / sum(l_extendedprice * (1 - l_discount)) as promo_revenue\nfrom\n    testdata.tpch.lineitem,\n    testdata.tpch.part\nwhere\n    l_partkey = p_partkey\n    and l_shipdate >= '1995-08-01'::DATE\n    and l_shipdate < '1995-09-01'::DATE;", 'Q15': "with revenue_cached as\n(select\n    l_suppkey as supplier_no,\n    sum(l_extendedprice * (1 - l_discount)) as total_revenue\nfrom\n    testdata.tpch.lineitem\nwhere\n    l_shipdate >= '1996-01-01'::DATE\n    and l_shipdate < '1996-04-01'::DATE\ngroup by l_suppkey)\n\n, max_revenue_cached as\n(select\n    max(total_revenue) as max_revenue\nfrom\n    revenue_cached)\n\nselect\n    s_suppkey,\n    s_name,\n    s_address,\n    s_phone,\n    total_revenue\nfrom\n    testdata.tpch.supplier,\n    revenue_cached,\n    max_revenue_cached\nwhere\n    s_suppkey = supplier_no\n    and total_revenue = max_revenue\norder by s_suppkey;", 'Q16': "select\n    p_brand,\n    p_type,\n    p_size,\n    count(distinct ps_suppkey) as supplier_cnt\nfrom\n    testdata.tpch.partsupp,\n    testdata.tpch.part\nwhere\n    p_partkey = ps_partkey\n    and p_brand <> 'Brand#34'\n    and p_type not like 'ECONOMY BRUSHED%'\n    and p_size in (22, 14, 27, 49, 21, 33, 35, 28)\n    and partsupp.ps_suppkey not in (\n        select\n            s_suppkey\n        from\n            testdata.tpch.supplier\n        where\n            s_comment like '%Customer%Complaints%'\n    )\ngroup by\n    p_brand,\n    p_type,\n    p_size\norder by\n    supplier_cnt desc,\n    p_brand,\n    p_type,\n    p_size;", 'Q17': "select\n    cast(sum(l_extendedprice) / 7.0 as decimal(32,2)) as avg_yearly\nfrom\n    testdata.tpch.lineitem,\n    testdata.tpch.part\nwhere\n    p_partkey = l_partkey\n    and p_brand = 'Brand#23'\n    and p_container = 'MED BOX'\n    and l_quantity < (\n        select\n            0.2 * avg(l_quantity)\n        from\n            testdata.tpch.lineitem\n        where\n            l_partkey = p_partkey\n    );", 'Q18': 'select\n    c_name,\n    c_custkey,\n    o_orderkey,\n    o_orderdate,\n    o_totalprice,\n    sum(l_quantity)\nfrom\n    testdata.tpch.customer,\n    testdata.tpch.orders,\n    testdata.tpch.lineitem\nwhere\n    o_orderkey in (\n        select\n            l_orderkey\n        from\n            testdata.tpch.lineitem\n        group by\n            l_orderkey having\n                sum(l_quantity) > 300\n    )\n    and c_custkey = o_custkey\n    and o_orderkey = l_orderkey\ngroup by\n    c_name,\n    c_custkey,\n    o_orderkey,\n    o_orderdate,\n    o_totalprice\norder by\n    o_totalprice desc,\n    o_orderdate\nlimit 100;', 'Q19': "select\n    sum(l_extendedprice* (1 - l_discount)) as revenue\nfrom\n    testdata.tpch.lineitem,\n    testdata.tpch.part\nwhere\n    (\n        p_partkey = l_partkey\n        and p_brand = 'Brand#32'\n        and p_container in ('SM CASE', 'SM BOX', 'SM PACK', 'SM PKG')\n        and l_quantity >= 7 and l_quantity <= 7 + 10\n        and p_size between 1 and 5\n        and l_shipmode in ('AIR', 'AIR REG')\n        and l_shipinstruct = 'DELIVER IN PERSON'\n    )\n    or\n    (\n        p_partkey = l_partkey\n        and p_brand = 'Brand#35'\n        and p_container in ('MED BAG', 'MED BOX', 'MED PKG', 'MED PACK')\n        and l_quantity >= 15 and l_quantity <= 15 + 10\n        and p_size between 1 and 10\n        and l_shipmode in ('AIR', 'AIR REG')\n        and l_shipinstruct = 'DELIVER IN PERSON'\n    )\n    or\n    (\n        p_partkey = l_partkey\n        and p_brand = 'Brand#24'\n        and p_container in ('LG CASE', 'LG BOX', 'LG PACK', 'LG PKG')\n        and l_quantity >= 26 and l_quantity <= 26 + 10\n        and p_size between 1 and 15\n        and l_shipmode in ('AIR', 'AIR REG')\n        and l_shipinstruct = 'DELIVER IN PERSON'\n    );", 'Q20': "select\n    s_name,\n    s_address\nfrom\n    testdata.tpch.supplier,\n    testdata.tpch.nation\nwhere\n    s_suppkey in (\n        select\n            ps_suppkey\n        from\n            testdata.tpch.partsupp\n        where\n            ps_partkey in (\n                select\n                    p_partkey\n                from\n                    testdata.tpch.part\n                where\n                    p_name like 'forest%'\n            )\n            and ps_availqty > (\n                select\n                    0.5 * sum(l_quantity)\n                from\n                    testdata.tpch.lineitem\n                where\n                    l_partkey = ps_partkey\n                    and l_suppkey = ps_suppkey\n                    and l_shipdate >= '1994-01-01'::DATE\n                    and l_shipdate < '1995-01-01'::DATE\n            )\n    )\n    and s_nationkey = n_nationkey\n    and n_name = 'CANADA'\norder by\n    s_name;", 'Q21': "select\n    s_name,\n    count(*) as numwait\nfrom\n    testdata.tpch.supplier,\n    testdata.tpch.lineitem l1,\n    testdata.tpch.orders,\n    testdata.tpch.nation\nwhere\n    s_suppkey = l1.l_suppkey\n    and o_orderkey = l1.l_orderkey\n    and o_orderstatus = 'F'\n    and l1.l_receiptdate > l1.l_commitdate\n    and exists (\n        select\n            *\n        from\n            testdata.tpch.lineitem l2\n        where\n            l2.l_orderkey = l1.l_orderkey\n            and l2.l_suppkey <> l1.l_suppkey\n    )\n    and not exists (\n        select\n            *\n        from\n            testdata.tpch.lineitem l3\n        where\n            l3.l_orderkey = l1.l_orderkey\n            and l3.l_suppkey <> l1.l_suppkey\n            and l3.l_receiptdate > l3.l_commitdate\n    )\n    and s_nationkey = n_nationkey\n    and n_name = 'SAUDI ARABIA'\ngroup by\n    s_name\norder by\n    numwait desc,\n    s_name\nlimit 100;", 'Q22': "with q22_customer_tmp_cached as\n(select\n    c_acctbal,\n    c_custkey,\n    substring(c_phone, 1, 2) as cntrycode\nfrom\n    testdata.tpch.customer\nwhere\n    substring(c_phone, 1, 2) = '13' or\n    substring(c_phone, 1, 2) = '31' or\n    substring(c_phone, 1, 2) = '23' or\n    substring(c_phone, 1, 2) = '29' or\n    substring(c_phone, 1, 2) = '30' or\n    substring(c_phone, 1, 2) = '18' or\n    substring(c_phone, 1, 2) = '17'\n),\n\nq22_customer_tmp1_cached as\n(select\n    avg(c_acctbal) as avg_acctbal\nfrom\n    q22_customer_tmp_cached\nwhere\n    c_acctbal > 0.00\n),\n\nq22_orders_tmp_cached as\n(select\n    o_custkey\nfrom\n    testdata.tpch.orders\ngroup by\n    o_custkey)\n\nselect\n    cntrycode,\n    count(1) as numcust,\n    sum(c_acctbal) as totacctbal\nfrom (\n    select\n        cntrycode,\n        c_acctbal,\n        avg_acctbal\n    from\n        q22_customer_tmp1_cached ct1 cross join (\n            select\n                cntrycode,\n                c_acctbal\n            from\n                q22_orders_tmp_cached ot\n                right outer join q22_customer_tmp_cached ct\n                on ct.c_custkey = ot.o_custkey\n            where\n                o_custkey is null\n        ) ct2\n) a\nwhere\n    c_acctbal > avg_acctbal\ngroup by\n    cntrycode\norder by\n    cntrycode;"}

def to_engine_sql(raw_sql: str, engine: str) -> str:
    if engine == "duckdb":
        return raw_sql.replace("testdata.tpch.", "")
    if engine == "opteryx":
        return raw_sql.replace("testdata.tpch.", "data.tpch.")
    raise ValueError(f"unknown engine: {engine}")

print(f"{len(QUERIES)} queries loaded: {list(QUERIES)}")

## 4. Benchmark runner

DuckDB gets one connection, reused across all queries, which is normal embedded-engine usage. Opteryx opens a fresh session per iteration and drains the result as morsels without rendering it. That matches how the [Opteryx performance suites](https://docs.opteryx.app/docs/reference/internals/benchmarking#method) run. Each query runs `ITERATIONS` warm iterations, with a garbage collection before each, and the **minimum** time is reported.

Opteryx leaves its own worker pools at their defaults, which it sizes to the machine. Pinning them higher than the runtime's core count makes it slower, not faster.

Row counts are recorded from both engines as a correctness check. Both engines query the same data, so they should return the same number of rows for every query.

In [ ]:
import gc
import time
import statistics


def bench_duckdb(con, sql, iterations):
    times = []
    rows = None
    for _ in range(iterations):
        gc.collect()
        t0 = time.perf_counter()
        result = con.execute(sql).fetchall()
        times.append(time.perf_counter() - t0)
        rows = len(result)
    return times, rows


def bench_opteryx(sql, iterations):
    times = []
    rows = None
    for _ in range(iterations):
        gc.collect()
        session = opteryx.session()
        try:
            t0 = time.perf_counter()
            row_count = 0
            for morsel in session.execute_to_morsels(sql):
                if morsel is not None and hasattr(morsel, "num_rows"):
                    row_count += morsel.num_rows
            times.append(time.perf_counter() - t0)
            rows = row_count
        finally:
            session.close()
    return times, rows

## 5. Run the benchmark

This cell takes the longest. It prints a line as each query finishes.

Both engines run a warm-up query first. The first query in a fresh process pays one-off costs: loading native libraries, starting thread pools, and the first read of each file from disk. On a new runtime that can take seconds. The Opteryx suites run the same warm-up and report it separately as the cold start, so it isn't counted against whichever query happens to run first.

In [ ]:
duckdb_con = duckdb.connect(DUCKDB_PATH, read_only=True)

t0 = time.perf_counter()
duckdb_con.execute("SELECT COUNT(*) FROM lineitem").fetchall()
print(f"DuckDB cold start:  {(time.perf_counter() - t0) * 1000:,.1f}ms")
t0 = time.perf_counter()
bench_opteryx("SELECT COUNT(*) FROM data.tpch.lineitem", 1)
print(f"Opteryx cold start: {(time.perf_counter() - t0) * 1000:,.1f}ms\n")

results = []  # rows of dicts, one per query

suite_start = time.perf_counter()
for name, raw_sql in QUERIES.items():
    row = {"query": name}
    for engine, bench_fn in (
        ("duckdb", lambda s: bench_duckdb(duckdb_con, s, ITERATIONS)),
        ("opteryx", lambda s: bench_opteryx(s, ITERATIONS)),
    ):
        sql = to_engine_sql(raw_sql, engine)
        try:
            times, rows = bench_fn(sql)
            row[f"{engine}_ms"] = min(times) * 1000
            row[f"{engine}_median_ms"] = statistics.median(times) * 1000
            row[f"{engine}_rows"] = rows
            row[f"{engine}_error"] = None
        except Exception as exc:
            row[f"{engine}_ms"] = None
            row[f"{engine}_median_ms"] = None
            row[f"{engine}_rows"] = None
            row[f"{engine}_error"] = f"{type(exc).__name__}: {exc}"
    results.append(row)
    parts = []
    for engine in ("duckdb", "opteryx"):
        v = row[f"{engine}_ms"]
        parts.append(f"{engine}={v:.1f}ms" if v is not None else f"{engine}=FAILED")
    print(f"{name}  " + "  ".join(parts))

duckdb_con.close()
print(f"\ndone in {time.perf_counter() - suite_start:.1f}s")

## 6. Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results).set_index("query")

# Row-count cross-check: the engines should agree, query by query.
row_cols = ["duckdb_rows", "opteryx_rows"]
mismatches = df[row_cols].apply(lambda r: len(set(v for v in r if v is not None)) > 1, axis=1)
if mismatches.any():
    print("ROW COUNT MISMATCH on:", list(df.index[mismatches]))
    display(df.loc[mismatches, row_cols])
else:
    print("Row counts agree across both engines on every query that ran.")

timing_cols = ["duckdb_ms", "opteryx_ms"]
display(df[timing_cols].round(1))

errors = df[["duckdb_error", "opteryx_error"]].dropna(how="all")
if not errors.empty:
    print("\nFailures:")
    display(errors)

compared = df[timing_cols].dropna()
print("Total time, queries both engines completed "
      f"({len(compared)}/{len(df)} queries):")
for engine, col in zip(("DuckDB", "Opteryx"), timing_cols):
    print(f"  {engine:<8} {compared[col].sum():>10,.1f} ms")

## 7. Charts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(df))
width = 0.38
colors = {"duckdb_ms": "#f0b429", "opteryx_ms": "#20bf6b"}
labels = {"duckdb_ms": "DuckDB (native)", "opteryx_ms": "Opteryx (skene)"}

for i, col in enumerate(timing_cols):
    values = df[col].fillna(0)
    offset = (i - (len(timing_cols) - 1) / 2) * width
    ax.bar(x + offset, values, width, label=labels[col], color=colors[col])

ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(df.index, rotation=45, ha="right")
ax.set_ylabel("min time (ms, log scale)")
ax.set_title(f"TPC-H SF{SCALE_FACTOR}: per-query min time across {ITERATIONS} warm runs")
ax.legend()
ax.grid(axis="y", alpha=0.3, which="both")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
engines = ["DuckDB", "Opteryx"]
values = [compared[c].sum() for c in timing_cols]
bar_colors = [colors[c] for c in timing_cols]
ax.bar(engines, values, color=bar_colors)
ax.set_ylabel("total time (ms)")
ax.set_title(f"TPC-H SF{SCALE_FACTOR}: total time, {len(compared)} comparable queries")
for i, v in enumerate(values):
    ax.text(i, v, f" {v:,.0f} ms", ha="center", va="bottom")
fig.tight_layout()
plt.show()

## Notes and caveats

- **Single process, single connection.** There is no concurrency and no cold-cache control. The OS page cache warms naturally after the first iteration of each query.
- **Acceleration structures.** DuckDB uses whatever `dbgen` produces by default. The `.skene` writer's `read_acceleration=True` adds zone maps, statistics and dictionary-ordered values, which is its default acceleration structure.
- **Allocator.** On Linux, the Opteryx suites preload the mimalloc build bundled with draken (`LD_PRELOAD`) to match production. A preload has to be set before Python starts, so this notebook runs with the system allocator.
- **Row-count cross-check.** The check in §6 catches query-translation bugs. If the engines disagree on row counts for a query, treat that query's timing as suspect.
- **Scaling up.** Raise `SCALE_FACTOR` in the configuration cell, for example to `10` for SF10. Generating and querying SF10 takes considerably longer, and it may exceed the memory of a free Colab runtime.
- **Not a published result.** As the [Benchmarking](https://docs.opteryx.app/docs/reference/internals/benchmarking#what-the-numbers-are-not) page says, numbers from one machine say nothing about another. This notebook exists so you can measure for yourself.